In [9]:
# Hücre 1: Kurulumlar, importlar, AMP, seed, Config

!pip install -q timm kagglehub seaborn

import os
import gc
import random
from pathlib import Path
import copy
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

# AMP ayarları
USE_AMP = torch.cuda.is_available()
if USE_AMP:
    from torch.cuda.amp import autocast, GradScaler
else:
    from contextlib import contextmanager

    @contextmanager
    def autocast():
        yield

    class GradScaler:
        def __init__(self, *args, **kwargs):
            pass

        def scale(self, loss):
            return loss

        def step(self, optimizer):
            optimizer.step()

        def update(self):
            pass


# Rastgelelikleri sabitle
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(42)


class Config:
    # Model
    MODEL_NAME = "swin_small_patch4_window7_224"
    # MODEL_NAME = "swin_tiny_patch4_window7_224"

    IMG_SIZE = 224
    NUM_CLASSES = 23  # mevcut 23 ama yeni sınıf eklersek degisecek

    BATCH_SIZE = 16
    EPOCHS = 40
    LEARNING_RATE = 3e-5       # backbone için LR
    WEIGHT_DECAY = 0.7        # regularization
    NUM_WORKERS = 2
    PATIENCE = 5               # Early stopping patience

    OUTPUT_DIR = Path("/kaggle/working")

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


print(f"Cihaz: {Config.DEVICE}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"AMP (mixed precision) kullanılacak mı? {USE_AMP}")


Cihaz: cuda
PyTorch Version: 2.6.0+cu124
CUDA Available: True
AMP (mixed precision) kullanılacak mı? True


In [10]:
# Hücre 2: DermNet datasetini indir, DataFrame oluştur, Train/Val split

print("\n" + "=" * 80)
print("DERMNET DATASETİ İNDİRİLİYOR...")
print("=" * 80)

path = kagglehub.dataset_download("shubhamgoel27/dermnet")
print("Path to dataset files:", path)

DATA_ROOT = Path(path)
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"

print("\nTrain klasörleri:", [d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
print("Test klasörleri :", [d.name for d in TEST_DIR.iterdir() if d.is_dir()])

IMG_EXT = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

# Sınıf isimleri ve label_map
class_names = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
label_map = {cls_name: idx for idx, cls_name in enumerate(class_names)}

print("\nSınıflar (label_map):")
for k, v in label_map.items():
    print(f"{v:2d} -> {k}")

Config.NUM_CLASSES = len(class_names)
print("\nNUM_CLASSES:", Config.NUM_CLASSES)


def build_split_df(split_dir: Path) -> pd.DataFrame:
    rows = []
    for cls_name in class_names:
        cls_dir = split_dir / cls_name
        if not cls_dir.exists():
            continue
        for fname in cls_dir.iterdir():
            if fname.is_file() and fname.suffix.lower() in IMG_EXT:
                rows.append(
                    {
                        "path": str(fname),
                        "label": cls_name,
                        "label_idx": label_map[cls_name],
                    }
                )
    return pd.DataFrame(rows)


full_train_df = build_split_df(TRAIN_DIR)
test_df = build_split_df(TEST_DIR)

print(f"\n Full train örnek sayısı: {len(full_train_df)}")
print(f"Test  örnek sayısı: {len(test_df)}")

print("\nOrijinal Train sınıf dağılımı:")
print(full_train_df["label"].value_counts())
print("\nTest sınıf dağılımı:")
print(test_df["label"].value_counts())

# Train'in %10'unu validation olarak ayır
train_df, val_df = train_test_split(
    full_train_df,
    test_size=0.1,
    stratify=full_train_df["label_idx"],
    random_state=42,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("\nYeni Train / Val bölünmesi:")
print(f"  Train set: {len(train_df)}")
print(f"  Val   set: {len(val_df)}")

print("\nTrain sınıf dağılımı (yeni):")
print(train_df["label"].value_counts())
print("\nVal sınıf dağılımı:")
print(val_df["label"].value_counts())



DERMNET DATASETİ İNDİRİLİYOR...
Path to dataset files: /kaggle/input/dermnet

Train klasörleri: ['Light Diseases and Disorders of Pigmentation', 'Lupus and other Connective Tissue diseases', 'Acne and Rosacea Photos', 'Systemic Disease', 'Poison Ivy Photos and other Contact Dermatitis', 'Vascular Tumors', 'Urticaria Hives', 'Atopic Dermatitis Photos', 'Bullous Disease Photos', 'Hair Loss Photos Alopecia and other Hair Diseases', 'Tinea Ringworm Candidiasis and other Fungal Infections', 'Psoriasis pictures Lichen Planus and related diseases', 'Melanoma Skin Cancer Nevi and Moles', 'Nail Fungus and other Nail Disease', 'Scabies Lyme Disease and other Infestations and Bites', 'Eczema Photos', 'Exanthems and Drug Eruptions', 'Herpes HPV and other STDs Photos', 'Seborrheic Keratoses and other Benign Tumors', 'Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions', 'Vasculitis Photos', 'Cellulitis Impetigo and other Bacterial Infections', 'Warts Molluscum and other Viral Infect

In [11]:
# Hücre 3: Dataset, transforms, model oluşturma fonksiyonu

class SkinDataset(Dataset):
    def __init__(self, dataframe, transform=None, max_retries=10):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.max_retries = max_retries

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Sonsuz recursion yerine kontrollü retry mekaniği
        for _ in range(self.max_retries):
            row = self.df.iloc[idx]
            img_path = row["path"]
            label = row["label_idx"]

            try:
                image = Image.open(img_path).convert("RGB")
                if self.transform:
                    image = self.transform(image)
                return image, torch.tensor(label, dtype=torch.long)
            except Exception:
                idx = random.randint(0, len(self.df) - 1)

        raise RuntimeError(f"Too many failed image loads. Last path: {img_path}")


# Train augmentations 
train_transforms = transforms.Compose(
    [
        transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.RandomVerticalFlip(p=0.3),
        # transforms.RandomAffine(
        #     degrees=0,
        #     translate=(0.03, 0.03),
        #     scale=(0.97, 1.03),
        #     shear=3,
        # ),
        # transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
        # transforms.ColorJitter(
        #     brightness=0.15,
        #     contrast=0.15,
        #     saturation=0.15,
        #     hue=0.05,
        # ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)

# Validation / Test için augmentationsız transform
val_transforms = transforms.Compose(
    [
        transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


def create_model():
    model = timm.create_model(
        Config.MODEL_NAME, pretrained=True, num_classes=Config.NUM_CLASSES
    )

    # Tüm parametreleri dondur
    for param in model.parameters():
        param.requires_grad = False

    # Son stage + head + norm katmanlarını fine-tune et
    for name, param in model.named_parameters():
        if (
            name.startswith("layers.3.")  # son stage
            or name.startswith("head.")
            or name.startswith("norm.")
        ):
            param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(
        f"Toplam: {total/1e6:.1f}M | Eğitilecek: {trainable/1e6:.1f}M parametre"
    )
    return model


In [12]:
# Hücre 4: Eğitim (train+val, early stopping), test, summary, zip

# Dataloaders
train_dataset = SkinDataset(train_df, transform=train_transforms)
val_dataset = SkinDataset(val_df, transform=val_transforms)
test_dataset = SkinDataset(test_df, transform=val_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    num_workers=Config.NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=Config.NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=Config.NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

# Loss için class_weights 
train_targets = train_df["label_idx"].values
class_sample_count = np.bincount(train_targets, minlength=Config.NUM_CLASSES)
class_weights_np = 1.0 / np.sqrt(class_sample_count + 1e-6)
class_weights = torch.tensor(class_weights_np, dtype=torch.float32).to(Config.DEVICE)

print("\nClass sample counts:", class_sample_count)
print("Class weights (loss):", class_weights.cpu().numpy())

print("\n" + "=" * 60)
print("TRAIN (90%) + VAL (10%) İLE FİNAL MODEL EĞİTİMİ (EARLY STOPPING: VAL LOSS)")
print("=" * 60)

final_model = create_model().to(Config.DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

backbone_params = [
    p
    for n, p in final_model.named_parameters()
    if "head" not in n and p.requires_grad
]
head_params = [
    p
    for n, p in final_model.named_parameters()
    if "head" in n and p.requires_grad
]

optimizer = optim.AdamW(
    [
        {"params": backbone_params, "lr": Config.LEARNING_RATE},           # lr
        {"params": head_params, "lr": Config.LEARNING_RATE * 5},         # 5 * lr
    ],
    weight_decay=Config.WEIGHT_DECAY,
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=Config.EPOCHS
)
scaler = GradScaler()

# Early stopping değişkenleri (VAL LOSS'A GÖRE)
best_val_loss = float("inf")
epochs_no_improve = 0
best_model_state = None
patience = Config.PATIENCE

print("\nFinal model eğitimi başlıyor (VAL LOSS'a göre EARLY STOPPING AKTİF)...")
for epoch in range(Config.EPOCHS):
    # ------------------------- TRAIN -------------------------
    final_model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{Config.EPOCHS} [TRAIN]",
    )
    for batch_idx, (images, targets) in enumerate(pbar):
        images, targets = images.to(Config.DEVICE), targets.to(Config.DEVICE)

        optimizer.zero_grad()

        if USE_AMP:
            with autocast():
                outputs = final_model(images)
                loss = criterion(outputs, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = final_model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

        batch_size = targets.size(0)
        train_loss += loss.item() * batch_size
        _, predicted = outputs.max(1)
        train_total += batch_size
        train_correct += predicted.eq(targets).sum().item()

        avg_loss = train_loss / train_total
        avg_acc = 100.0 * train_correct / train_total
        pbar.set_postfix({"train_loss": f"{avg_loss:.4f}", "train_acc": f"{avg_acc:.2f}%"})

    epoch_train_loss = train_loss / train_total
    epoch_train_acc = 100.0 * train_correct / train_total

    # ------------------------- VALIDATION -------------------------
    final_model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(Config.DEVICE), targets.to(Config.DEVICE)

            if USE_AMP:
                with autocast():
                    outputs = final_model(images)
                    loss = criterion(outputs, targets)
            else:
                outputs = final_model(images)
                loss = criterion(outputs, targets)

            batch_size = targets.size(0)
            val_loss += loss.item() * batch_size
            _, predicted = outputs.max(1)
            val_total += batch_size
            val_correct += predicted.eq(targets).sum().item()

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = 100.0 * val_correct / val_total

    scheduler.step()

    print(
        f"Epoch [{epoch+1}/{Config.EPOCHS}] "
        f"- Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}% "
        f"| Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%"
    )

    # -------- EARLY STOPPING KONTROLÜ (VAL LOSS'A GÖRE) --------
    if epoch_val_loss < best_val_loss - 1e-6:  # min decrease
        best_val_loss = epoch_val_loss
        epochs_no_improve = 0
        best_model_state = copy.deepcopy(final_model.state_dict())
        print(f"  -> Yeni en iyi VAL LOSS: {best_val_loss:.4f}")
    else:
        epochs_no_improve += 1
        print(
            f"  -> Val loss iyileşmedi ({epochs_no_improve}/{patience}) "
            f"(best val loss: {best_val_loss:.4f})"
        )
        if epochs_no_improve >= patience:
            print(
                f"\nEarly stopping tetiklendi! Epoch {epoch+1}'da duruldu. "
                f"En iyi val loss: {best_val_loss:.4f}"
            )
            break

# En iyi modeli geri yükle 
if best_model_state is not None:
    final_model.load_state_dict(best_model_state)

# Final modeli kaydet
final_model_path = Config.OUTPUT_DIR / "final_model.pth"
torch.save(final_model.state_dict(), final_model_path)
print(f"\nFinal model kaydedildi: {final_model_path}")


# ============================ TEST SETİ DEĞERLENDİRMESİ ============================
print("\n" + "=" * 60)
print("TEST SETİ DEĞERLENDİRMESİ")
print("=" * 60)

final_model.eval()
test_preds = []
test_targets = []

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="Test Evaluation"):
        images = images.to(Config.DEVICE)

        if USE_AMP:
            with autocast():
                outputs = final_model(images)
        else:
            outputs = final_model(images)

        _, predicted = outputs.max(1)
        test_preds.extend(predicted.cpu().numpy())
        test_targets.extend(targets.numpy())

test_acc = accuracy_score(test_targets, test_preds)
test_f1 = f1_score(test_targets, test_preds, average="weighted")

print(f"\n FINAL TEST ACCURACY: {test_acc*100:.2f}%")
print(f" FINAL TEST F1-SCORE (Weighted): {test_f1:.4f}")


# ============================ ÖZET RAPOR ============================
total_images = len(full_train_df) + len(test_df)

print("\n" + "=" * 80)
print(" " * 25 + "FINAL SUMMARY REPORT")
print("=" * 80)

print("\n DATASET STATISTICS:")
print(f"  • Total images: {total_images}")
print(
    f"  • Train set (eğitim için): {len(train_df)} "
    f"({len(train_df)/total_images*100:.1f}%)"
)
print(
    f"  • Val set: {len(val_df)} "
    f"({len(val_df)/total_images*100:.1f}%)"
)
print(
    f"  • Test set: {len(test_df)} "
    f"({len(test_df)/total_images*100:.1f}%)"
)
print(f"  • Number of classes: {Config.NUM_CLASSES}")

print("\n MODEL CONFIGURATION:")
print(f"  • Architecture: {Config.MODEL_NAME}")
print(f"  • Image size: {Config.IMG_SIZE}x{Config.IMG_SIZE}")
print(f"  • Batch size: {Config.BATCH_SIZE}")
print(f"  • Learning rate: {Config.LEARNING_RATE}")
print(f"  • Weight decay: {Config.WEIGHT_DECAY}")
print(f"  • Max epochs: {Config.EPOCHS}")
print(f"  • Early stopping patience: {Config.PATIENCE}")

print("\n FINAL PERFORMANCE:")
print(f"  • Test accuracy: {test_acc*100:.2f}%")
print(f"  • Test F1-score (weighted): {test_f1:.4f}")
print(f"  • Test samples evaluated: {len(test_targets)}")

print("\n SAVED FILES:")
print(f"  • final_model.pth - Early stopping ile seçilen en iyi model")


# ============================ MODELİ ZİPLEYİP KAYDET ============================
import zipfile

zip_path = Config.OUTPUT_DIR / "final_model.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(final_model_path, arcname="final_model.pth")

print(f"\nZipped model: {zip_path}")
print("Kaggle'da bu .zip dosyasını 'Output files' kısmından indirebilirsin.")


print("\n" + "=" * 80)
print(" " * 20 + " PIPELINE COMPLETED SUCCESSFULLY!")
print("=" * 80)



Class sample counts: [ 756 1034  440  403  259 1112  364  215  365  511  378  417  936  234
 1264  388 1234  545 1170  191  434  374  977]
Class weights (loss): [0.03636965 0.03109852 0.04767313 0.04981355 0.06213698 0.02998801
 0.05241424 0.06819943 0.05234239 0.04423739 0.05143445 0.04897021
 0.03268602 0.06537204 0.0281272  0.05076731 0.02846705 0.0428353
 0.02923527 0.07235746 0.04800154 0.05170877 0.03199283]

TRAIN (90%) + VAL (10%) İLE FİNAL MODEL EĞİTİMİ (EARLY STOPPING: VAL LOSS)


/tmp/ipykernel_47/2434713629.py:70: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Toplam: 48.9M | Eğitilecek: 15.4M parametre

Final model eğitimi başlıyor (VAL LOSS'a göre EARLY STOPPING AKTİF)...


Epoch 1/40 [TRAIN]:   0%|          | 0/876 [00:00<?, ?it/s]/tmp/ipykernel_47/2434713629.py:96: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1/40 [TRAIN]: 100%|██████████| 876/876 [01:07<00:00, 12.94it/s, train_loss=2.5025, train_acc=32.48%]
/tmp/ipykernel_47/2434713629.py:132: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/40] - Train Loss: 2.5025, Train Acc: 32.48% | Val Loss: 2.1862, Val Acc: 41.65%
  -> Yeni en iyi VAL LOSS: 2.1862


...

Epoch [39/40] - Train Loss: 0.7201, Train Acc: 91.71% | Val Loss: 1.4252, Val Acc: 68.19%
  -> Val loss iyileşmedi (2/5) (best val loss: 1.4236)


Epoch 40/40 [TRAIN]: 100%|██████████| 876/876 [01:06<00:00, 13.12it/s, train_loss=0.7266, train_acc=91.37%]


Epoch [40/40] - Train Loss: 0.7266, Train Acc: 91.37% | Val Loss: 1.4250, Val Acc: 68.12%
  -> Val loss iyileşmedi (3/5) (best val loss: 1.4236)

Final model kaydedildi: /kaggle/working/final_model.pth

TEST SETİ DEĞERLENDİRMESİ


Test Evaluation:   0%|          | 0/251 [00:00<?, ?it/s]/tmp/ipykernel_47/2434713629.py:199: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Test Evaluation: 100%|██████████| 251/251 [00:18<00:00, 13.55it/s]



 FINAL TEST ACCURACY: 69.54%
 FINAL TEST F1-SCORE (Weighted): 0.6953

                         FINAL SUMMARY REPORT

 DATASET STATISTICS:
  • Total images: 19559
  • Train set (eğitim için): 14001 (71.6%)
  • Val set: 1556 (8.0%)
  • Test set: 4002 (20.5%)
  • Number of classes: 23

 MODEL CONFIGURATION:
  • Architecture: swin_small_patch4_window7_224
  • Image size: 224x224
  • Batch size: 16
  • Learning rate: 3e-05
  • Weight decay: 0.7
  • Max epochs: 40
  • Early stopping patience: 5

 FINAL PERFORMANCE:
  • Test accuracy: 69.54%
  • Test F1-score (weighted): 0.6953
  • Test samples evaluated: 4002

 SAVED FILES:
  • final_model.pth - Early stopping ile seçilen en iyi model

Zipped model: /kaggle/working/final_model.zip
Kaggle'da bu .zip dosyasını 'Output files' kısmından indirebilirsin.

                     PIPELINE COMPLETED SUCCESSFULLY!
